# Transformación de datos por tipo — resumen para Scikit-learn

Guía de **decisión rápida**: dado el tipo de cada columna en pandas, ¿qué transformación aplicar para que un modelo de scikit-learn pueda entrenar?

| Notebook relacionado | Qué cubre |
|----------------------|-----------|
| [04.03](../04-pandas/04.03-pandas-data-manipulation.ipynb) | `fillna`, `replace`, `get_dummies` en pandas |
| [07.02](07.02-scikit-learn-preprocessing.ipynb) | API de cada transformador (`StandardScaler`, `OneHotEncoder`…) |
| [07.06](07.06-scikit-learn-pipelines.ipynb) | `Pipeline` + `ColumnTransformer` sin data leakage |
| [07.a](../07.a-esquemas-supervisados/) | Tratamiento **manual** en pandas antes del modelo |

> **¿Por qué aquí y no en 04-pandas?** Porque el objetivo final es alimentar **estimadores sklearn**; pandas es la herramienta de inspección y limpieza inicial.

## Qué exige scikit-learn

La mayoría de modelos sklearn esperan:

1. **`X`**: matriz 2D **numérica** (`float` o `int`), **sin NaN**.
2. **`y`**: vector 1D (numérico en regresión; entero o string en clasificación según el estimador).
3. **Mismo número de columnas** en train, val y test (one-hot fija el esquema en `fit`).

**Regla de oro:** aprende estadísticos y categorías solo con **train** (`fit` / `fit_transform` en train; `transform` en val/test). Ver [07.02](07.02-scikit-learn-preprocessing.ipynb).

## Tabla resumen por tipo de dato

| Tipo en pandas | Ejemplos | Problemas habituales | Transformación típica | Sklearn |
|----------------|----------|----------------------|---------------------|---------|
| **Numérico** (`int64`, `float64`) | edad, precio, score | NaN, outliers, escalas distintas | Imputar → escalar (a veces log) | `SimpleImputer` + `StandardScaler` / `RobustScaler` |
| **Categórico nominal** (`object`, `category` sin orden) | ciudad, color, producto | NaN, muchas categorías | Imputar → one-hot | `SimpleImputer` + `OneHotEncoder` |
| **Categórico ordinal** (orden lógico) | talla S/M/L, nivel bajo/medio/alto | NaN, orden incorrecto | Imputar → entero ordenado | `OrdinalEncoder(categories=[[...]])` |
| **Booleano** (`bool`) | activo, acepta_terminos | NaN raro | `astype(int)` 0/1 | A menudo ya válido; si hay NaN → imputer |
| **Fecha/hora** (`datetime64`) | fecha_compra | No lo entiende el modelo directamente | Extraer año, mes, día… o descartar | `FunctionTransformer` o pandas → luego numérico |
| **Texto libre** | comentario, reseña | Alta cardinalidad | TF-IDF, embeddings (fuera del MVP) | `CountVectorizer` / otro módulo |
| **Target `y` (clasificación)** | spam/no, clase | Clases desbalanceadas | Codificar a enteros si hace falta | `LabelEncoder` solo en **y** (no en X) |
| **Target `y` (regresión)** | precio, temperatura | NaN en y | Eliminar filas o imputar antes del split | Sin encoder; debe ser numérico |

**Escalado:** casi obligatorio con SVM, KNN, redes; recomendable con regresión lineal; **no** hace falta con árboles (RandomForest, XGBoost…).

## Dataset de ejemplo (mixto)

DataFrame con varios tipos para practicar el flujo de inspección → transformación.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "edad": [25, 30, np.nan, 35, 28],
    "ingresos": [30000, 45000, 50000, 40000, 38000],
    "activo": [True, False, True, True, False],
    "zona": ["Centro", "Periferia", "Centro", np.nan, "Periferia"],
    "talla": ["S", "M", "L", "M", "S"],
    "fecha_alta": pd.to_datetime(["2020-01-15", "2019-06-01", "2021-03-20", "2018-11-10", "2022-07-01"]),
    "compró": ["si", "no", "si", "no", "si"],
})
y = df.pop("compró")  # target de clasificación binaria
display(df)
print("Target y:", y.tolist())

,edad,ingresos,activo,zona,talla,fecha_alta
0,25.0,30000,True,Centro,S,2020-01-15
1,30.0,45000,False,Periferia,M,2019-06-01
2,NaN,50000,True,Centro,L,2021-03-20
3,35.0,40000,True,NaN,M,2018-11-10
4,28.0,38000,False,Periferia,S,2022-07-01


Target y: ['si', 'no', 'si', 'no', 'si']


## 1. Inspeccionar antes de transformar

Siempre: `dtypes`, faltantes y separar columnas por tipo con `select_dtypes`.

In [2]:
print("Tipos:\n", df.dtypes)
print("\nFaltantes:\n", df.isna().sum())

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
date_cols = df.select_dtypes(include=["datetime64"]).columns.tolist()

print("\nNuméricas:", num_cols)
print("Categóricas:", cat_cols)
print("Booleanas:", bool_cols)
print("Fechas:", date_cols)

Tipos:
 edad                 float64
ingresos               int64
activo                  bool
zona                  object
talla                 object
fecha_alta    datetime64[ns]
dtype: object

Faltantes:
 edad          1
ingresos      0
activo        0
zona          1
talla         0
fecha_alta    0
dtype: int64

Numéricas: ['edad', 'ingresos']
Categóricas: ['zona', 'talla']
Booleanas: ['activo']
Fechas: ['fecha_alta']


## 2. Columnas numéricas

| Paso | Pandas (manual, estilo 07.a) | Sklearn (recomendado en pipeline) |
|------|------------------------------|-------------------------------------|
| Faltantes | `df["col"].fillna(df["col"].median())` | `SimpleImputer(strategy="median")` |
| Escalado | `(x - mean) / std` (solo stats de train) | `StandardScaler()` o `RobustScaler()` si hay outliers |

> Imputar con la **mediana** suele ser más robusta que la media ante outliers.

In [3]:
# --- Pandas (demo; en proyecto real: stats solo de train) ---
df_num = df[num_cols].copy()
for col in num_cols:
    df_num[col] = df_num[col].fillna(df_num[col].median())
display(df_num)

# --- Sklearn (mismo resultado, encaja en Pipeline) ---
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer_num = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_num = scaler.fit_transform(imputer_num.fit_transform(df[num_cols]))
print("Shape numérico escalado:", X_num.shape)
print(X_num.round(2))

,edad,ingresos
0,25.0,30000
1,30.0,45000
2,29.0,50000
3,35.0,40000
4,28.0,38000


Shape numérico escalado: (5, 2)
[[-1.35 -1.57]
 [ 0.18  0.65]
 [-0.12  1.39]
 [ 1.72 -0.09]
 [-0.43 -0.39]]


## 3. Categóricas nominales (sin orden)

Ejemplo: `zona` (Centro / Periferia). **No** uses enteros arbitrarios (Centro=0, Periferia=1) salvo que el modelo lo exija y no haya orden.

| Paso | Pandas | Sklearn |
|------|--------|---------|
| Faltantes | `fillna("desconocido")` | `SimpleImputer(strategy="most_frequent")` o constante |
| Codificar | `pd.get_dummies(..., drop_first=True)` | `OneHotEncoder(drop="first", handle_unknown="ignore")` |

In [4]:
nominal_cols = ["zona"]

# Pandas
df_cat = df[nominal_cols].fillna("desconocido")
dummies = pd.get_dummies(df_cat, columns=nominal_cols, drop_first=True)
display(dummies)

# Sklearn
from sklearn.preprocessing import OneHotEncoder

imputer_cat = SimpleImputer(strategy="most_frequent")
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
X_cat = ohe.fit_transform(imputer_cat.fit_transform(df[nominal_cols]))
print("Columnas one-hot:", ohe.get_feature_names_out(nominal_cols))

,zona_Periferia,zona_desconocido
0,False,False
1,True,False
2,False,False
3,False,True
4,True,False


Columnas one-hot: ['zona_Periferia']


## 4. Categóricas ordinales (con orden)

Ejemplo: `talla` S < M < L. Aquí **sí** tiene sentido un entero ordenado, pero **tú defines el orden** (no alfabético automático).

In [5]:
orden_talla = [["S", "M", "L"]]

# Pandas
mapa = {"S": 0, "M": 1, "L": 2}
df["talla_ord"] = df["talla"].map(mapa)  # demo

# Sklearn (orden explícito en fit)
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=orden_talla)
X_ord = oe.fit_transform(df[["talla"]])
print("Ordinal:", X_ord.ravel())

Ordinal: [0. 1. 2. 1. 0.]


## 5. Booleanos

Convertir a `0/1` suele bastar. Si hay NaN, imputar antes (p. ej. moda).

In [6]:
df[bool_cols].astype(int)

,activo
0,1
1,0
2,1
3,1
4,0


## 6. Fechas y horas

Sklearn no usa `datetime64` directamente. Extrae componentes numéricos o elimina la columna si no aporta.

In [7]:
df_dates = df[date_cols].copy()
for col in date_cols:
    df_dates[f"{col}_año"] = df_dates[col].dt.year
    df_dates[f"{col}_mes"] = df_dates[col].dt.month
df_dates = df_dates.drop(columns=date_cols)
display(df_dates)

,fecha_alta_año,fecha_alta_mes
0,2020,1
1,2019,6
2,2021,3
3,2018,11
4,2022,7


## 7. Target `y`

- **Regresión:** `y` numérico, sin NaN (elimina filas con target faltante).
- **Clasificación:** muchos estimadores aceptan strings; otros piden enteros → `LabelEncoder` **solo en y**.
- **Nunca** uses `LabelEncoder` en columnas de `X` con varias categorías mezcladas (usa `OneHotEncoder`).

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Clases:", le.classes_)
print("y codificado:", y_encoded)

Clases: ['no' 'si']
y codificado: [1 0 1 0 1]


## 8. Dos caminos hacia el modelo

| Enfoque | Cuándo | Dónde verlo |
|---------|--------|-------------|
| **Manual (pandas)** | Aprender el flujo, datasets pequeños, MVP | [07.a](../07.a-esquemas-supervisados/01-regresion-lineal.ipynb) |
| **Pipeline + ColumnTransformer** | Proyectos reales, evitar leakage, CV | [07.06](07.06-scikit-learn-pipelines.ipynb) |

En ambos casos el **criterio por tipo de dato** de la tabla superior es el mismo; cambia **dónde** encadenas los pasos.

## 9. Ejemplo integrado con `ColumnTransformer`

Un solo preprocesador para columnas numéricas, nominales y ordinales. Luego encaja en un `Pipeline` con el modelo.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_cols = ["edad", "ingresos"]
nominal_cols = ["zona"]
ordinal_cols = ["talla"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("nominal", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
    ]), nominal_cols),
    ("ordinal", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(categories=[["S", "M", "L"]])),
    ]), ordinal_cols),
], remainder="drop")  # descarta columnas no listadas (p. ej. fechas sin transformar)

X_ready = preprocessor.fit_transform(df)
print("Shape final para sklearn:", X_ready.shape)
print("Nombres:", preprocessor.get_feature_names_out())

Shape final para sklearn: (5, 4)
Nombres: ['num__edad' 'num__ingresos' 'nominal__zona_Periferia' 'ordinal__talla']


## Checklist antes de `model.fit(X, y)`

- [ ] Separaste **X** y **y**; el target no está en las features.
- [ ] Hiciste **split** train/val/test (estratificado si clasificación desbalanceada).
- [ ] Cada columna tiene asignado un tratamiento según su **tipo** (tabla resumen).
- [ ] **Sin NaN** en `X` tras imputación.
- [ ] **Solo numéricos** en `X` (bool/fecha/texto ya transformados).
- [ ] Preprocesado: `fit_transform` en **train**, `transform` en val/test.
- [ ] Mismo `random_state` y mismas columnas en todos los conjuntos.

Siguiente paso: [07.06 — Pipelines](07.06-scikit-learn-pipelines.ipynb) o [07.a — esquema manual](../07.a-esquemas-supervisados/01-regresion-lineal.ipynb).